# PaddleOCR Prescription Recognition - Fine-Tuning on Google Colab

This notebook fine-tunes **PP-OCRv4** English recognition model on custom prescription data.

**Steps:**
1. Install PaddlePaddle GPU + PaddleOCR
2. Clone PaddleOCR repo (training tools)
3. Generate synthetic training data
4. Download pretrained model
5. Train recognition model
6. Export & download trained model

> **Runtime:** Go to `Runtime > Change runtime type > GPU (T4)` before running!

## 1. Setup Environment

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Install PaddlePaddle GPU and PaddleOCR
!pip install paddlepaddle-gpu==2.6.2 -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install paddleocr==2.7.3
!pip install PyYAML Pillow numpy

In [ ]:
# Verify installation
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.device.is_compiled_with_cuda()}")
if paddle.device.is_compiled_with_cuda():
    print(f"GPU count: {paddle.device.cuda.device_count()}")

## 2. Clone PaddleOCR Repository

In [ ]:
import os
WORK_DIR = '/content/prescription-ocr'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Clone PaddleOCR repo (training tools)
if not os.path.exists('PaddleOCR'):
    !git clone --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
    !pip install -r PaddleOCR/requirements.txt
else:
    print("PaddleOCR repo already cloned")

## 3. Generate Training Data

Generates ~8000 line-level text images from 500 synthetic prescriptions with 3 quality levels.

In [ ]:
import random
from pathlib import Path
from datetime import datetime, timedelta
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import numpy as np

# Configuration
NUM_PRESCRIPTIONS = 500
LINE_IMAGE_WIDTH = 800
LINE_IMAGE_HEIGHT = 60
QUALITY_WEIGHTS = [0.50, 0.30, 0.20]  # high, medium, low

# ==================== PRESCRIPTION DATA ====================

PATIENT_NAMES = [
    "John Silva", "Maria Perera", "Kasun Fernando", "Dilini Rajapaksa",
    "Nimal De Silva", "Sanduni Wickramasinghe", "Chaminda Jayawardena",
    "Rushika Gunasekara", "Thilini Dissanayake", "Nuwan Bandara",
    "Hasini Samaraweera", "Rohan Gamage", "Savithri Herath",
    "Prasanna Mendis", "Ayesha Pathirana", "Lakmal Rathnayake",
    "Ishara Rodrigo", "Buddhika Amarasinghe", "Shanika Kumarasinghe",
    "Malith Jayakody", "Nadeesha Wickremasinghe", "Udara Gamage", "Yasmin Fonseka",
    "Sameera Pathirana", "Janaki Munasinghe", "Kusal Rajapakse", "Dilrukshi Senanayake",
    "Asanka Herath", "Minoli Samarasinghe", "Gayan Liyanage", "Rashmi Wanninayake",
    "Manoj Gunathilake", "Thanuja Ratnayake", "Sachini Dharmasena", "Buddhika Wijesekara",
    "Hasini Kudagama", "Danushka Perera", "Amaya Cooray", "Upul Mendis",
    "Nethmi Karunaratne", "Charith Atapattu", "Madushani Ekanayake", "Dinesh Madushanka",
    "Shashika Priyankara", "Chaminda Rajakaruna", "Gayani Kapukotuwa", "Nuwan Pradeep",
    "Tharanga Vithanage", "Senali Weerasinghe", "Kavindya Gunaratne", "Chatura Alwis",
    "Dimuthu Karunaratne", "Niluka Fernando", "Prabath Nissanka", "Shalini Kodagoda",
    "Anuradha Jayalath", "Hemantha Silva", "Priyani Wijesinghe", "Sanjeewa Cooray",
    "Rajesh Kumar", "Priya Sharma", "Amit Patel", "Sneha Reddy", "Vikram Singh",
    "Anjali Gupta", "Suresh Nair", "Kavita Menon", "Arjun Krishnan", "Deepa Iyer",
    "David Anderson", "Sarah Johnson", "Michael Chen", "Emily Williams", "James Brown",
    "Lisa Davis", "Robert Martinez", "Jennifer Garcia", "William Rodriguez", "Mary Wilson",
    "Ahmed Hassan", "Fatima Ali", "Omar Abdullah", "Aisha Khan", "Mohammed Rahman",
    "Li Wei", "Zhang Mei", "Wang Jian", "Tanaka Hiroshi", "Kim Min-Jun",
]

DOCTOR_NAMES = [
    "Dr. A. Fernando", "Dr. K. Perera", "Dr. S. Silva",
    "Dr. M. Gunawardena", "Dr. R. Jayasinghe", "Dr. N. Dissanayake",
    "Dr. P. Wickramasinghe", "Dr. T. Ratnayake",
    "Dr. Chandana Wijesuriya", "Dr. Lakshmi Fernando", "Dr. Ajith Perera",
    "Dr. Suresh Bandara", "Dr. Nirmala Silva", "Dr. Rohan Jayasuriya",
    "Dr. Priyanka De Silva", "Dr. Mahesh Rathnayake", "Dr. Sampath Kumarasinghe",
    "Dr. Anoma Jayawardena", "Dr. Nishan Gunaratne", "Dr. Buddhika Wickramasinghe",
    "Dr. Chamari Amarasena", "Dr. Ruwan Samarakoon", "Dr. Manjula Seneviratne",
    "Dr. Sarah Chen", "Dr. Michael Thompson", "Dr. Priya Sharma",
    "Dr. David Anderson", "Dr. Emily Williams", "Dr. Ahmed Hassan",
]

HOSPITALS = [
    "Lanka Hospital", "Nawaloka Hospital", "Asiri Medical Center",
    "Durdans Hospital", "Apollo Hospital", "Central Hospital",
    "City Medical Center", "National Hospital",
    "Asiri Central Hospital", "Hemas Hospital", "Oasis Hospital",
    "Ninewells Hospital", "Golden Key Hospital", "Royal Hospital",
    "National Hospital of Sri Lanka", "Lady Ridgeway Hospital",
    "General Hospital Kandy", "Teaching Hospital Karapitiya",
    "St. Michael's Medical Center", "Metro Healthcare Hospital",
    "Prime Care Hospital", "Sunshine Medical Center", "Greenfield Hospital",
    "Riverside Medical Center", "Valley View Hospital", "Summit Healthcare",
    "Lakeside Medical Center", "Harbor Medical Hospital", "Clearwater Hospital",
]

MEDICINES = [
    {"name": "Paracetamol", "dosages": ["500mg", "650mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Amoxicillin", "dosages": ["250mg", "500mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Omeprazole", "dosages": ["20mg", "40mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Metformin", "dosages": ["500mg", "1000mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Amlodipine", "dosages": ["5mg", "10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Atorvastatin", "dosages": ["10mg", "20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Aspirin", "dosages": ["75mg", "100mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Cetirizine", "dosages": ["10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Azithromycin", "dosages": ["250mg", "500mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Losartan", "dosages": ["25mg", "50mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Vitamin D3", "dosages": ["1000IU", "2000IU"], "forms": ["Capsule", "Cap"]},
    {"name": "Folic Acid", "dosages": ["5mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Ibuprofen", "dosages": ["200mg", "400mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Ciprofloxacin", "dosages": ["250mg", "500mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Doxycycline", "dosages": ["100mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Simvastatin", "dosages": ["10mg", "20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Lisinopril", "dosages": ["5mg", "10mg", "20mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Levothyroxine", "dosages": ["50mcg", "100mcg"], "forms": ["Tablet", "Tab"]},
    {"name": "Pantoprazole", "dosages": ["20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Diclofenac", "dosages": ["25mg", "50mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Gabapentin", "dosages": ["100mg", "300mg"], "forms": ["Capsule", "Cap"]},
    {"name": "Furosemide", "dosages": ["20mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Warfarin", "dosages": ["2mg", "5mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Propranolol", "dosages": ["10mg", "40mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Clarithromycin", "dosages": ["250mg", "500mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Montelukast", "dosages": ["4mg", "10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Clopidogrel", "dosages": ["75mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Rosuvastatin", "dosages": ["5mg", "10mg", "20mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Fexofenadine", "dosages": ["120mg", "180mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Metoprolol", "dosages": ["25mg", "50mg", "100mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Sertraline", "dosages": ["50mg", "100mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Prednisolone", "dosages": ["5mg", "10mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Salbutamol", "dosages": ["2mg", "4mg"], "forms": ["Tablet", "Inhaler"]},
    {"name": "Calcium Carbonate", "dosages": ["500mg", "1000mg"], "forms": ["Tablet", "Tab"]},
    {"name": "Tramadol", "dosages": ["50mg", "100mg"], "forms": ["Tablet", "Tab"]},
]

FREQUENCIES = [
    "1-0-1 (Twice daily)", "1-1-1 (Three times daily)",
    "0-0-1 (Once daily at night)", "1-0-0 (Once daily in morning)",
    "Twice daily", "Three times daily", "As needed",
    "Every 6 hours", "Every 8 hours", "Every 12 hours",
    "At bedtime", "Before meals", "After meals",
]

DURATIONS = [
    "5 days", "7 days", "10 days", "14 days", "1 month",
    "3 months", "3 days", "21 days", "2 weeks", "30 days",
]

print("Data definitions loaded!")
print(f"  Patients: {len(PATIENT_NAMES)}, Doctors: {len(DOCTOR_NAMES)}")
print(f"  Hospitals: {len(HOSPITALS)}, Medicines: {len(MEDICINES)}")

In [ ]:
# Font loading (Colab uses Linux fonts)
def load_fonts():
    font_families = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
    ]
    # Install extra fonts for more diversity
    os.system('apt-get install -y fonts-liberation fonts-dejavu fonts-freefont-ttf fonts-ubuntu > /dev/null 2>&1')
    extra = [
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSerif.ttf",
        "/usr/share/fonts/truetype/freefont/FreeMono.ttf",
        "/usr/share/fonts/truetype/ubuntu/Ubuntu-R.ttf",
        "/usr/share/fonts/truetype/ubuntu/UbuntuMono-R.ttf",
    ]
    font_families.extend(extra)

    sizes = {'large': 36, 'medium': 28, 'normal': 24, 'small': 20}
    available_fonts = []
    for font_path in font_families:
        try:
            ImageFont.truetype(font_path, 24)
            available_fonts.append(font_path)
        except OSError:
            continue

    if not available_fonts:
        print("WARNING: No fonts found, using default")
        return [{key: ImageFont.load_default() for key in sizes}]

    print(f"Found {len(available_fonts)} fonts")
    all_font_sets = []
    for font_name in available_fonts:
        font_set = {}
        for key, size in sizes.items():
            try:
                font_set[key] = ImageFont.truetype(font_name, size)
            except OSError:
                font_set[key] = ImageFont.load_default()
        all_font_sets.append(font_set)
    return all_font_sets

# Image rendering helpers
def random_date():
    days_ago = random.randint(0, 30)
    d = datetime.now() - timedelta(days=days_ago)
    fmt = random.choice(["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d"])
    return d.strftime(fmt)

def render_line(text, font, width=LINE_IMAGE_WIDTH, height=LINE_IMAGE_HEIGHT):
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    y_offset = random.randint(5, max(6, height - 40))
    x_offset = random.randint(10, 40)
    draw.text((x_offset, y_offset), text, fill='black', font=font)
    return img

def degrade_image(img, quality):
    if quality == 'high':
        if random.random() > 0.5:
            factor = random.uniform(0.9, 1.1)
            img = ImageEnhance.Brightness(img).enhance(factor)
        return img
    if quality == 'medium':
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
        angle = random.uniform(-2, 2)
        img = img.rotate(angle, fillcolor='white', expand=False)
        arr = np.array(img)
        noise = np.random.normal(0, 5, arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)
    # low quality
    img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(1.5, 3.0)))
    angle = random.uniform(-4, 4)
    img = img.rotate(angle, fillcolor='white', expand=False)
    arr = np.array(img)
    noise = np.random.normal(0, 12, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def pick_quality():
    return random.choices(['high', 'medium', 'low'], weights=QUALITY_WEIGHTS)[0]

print("Helper functions defined!")

In [ ]:
# Generate the full dataset
TRAINING_DATA_DIR = Path(WORK_DIR) / 'training_data'
train_dir = TRAINING_DATA_DIR / 'train'
val_dir = TRAINING_DATA_DIR / 'val'

for d in [train_dir, val_dir]:
    if d.exists():
        for f in d.glob('*.png'):
            f.unlink()
    d.mkdir(parents=True, exist_ok=True)

font_sets = load_fonts()
all_samples = []
line_count = 0
quality_counts = {'high': 0, 'medium': 0, 'low': 0}

print(f"\nGenerating data from {NUM_PRESCRIPTIONS} prescriptions...")

for rx_idx in range(1, NUM_PRESCRIPTIONS + 1):
    patient = random.choice(PATIENT_NAMES)
    doctor = random.choice(DOCTOR_NAMES)
    hospital = random.choice(HOSPITALS)
    date = random_date()
    age = random.randint(18, 80)
    num_meds = random.randint(1, 5)

    lines = []
    lines.append((hospital, 'large'))
    lines.append((f"Patient Name: {patient}", 'normal'))
    lines.append((f"Age: {age} years", 'normal'))
    lines.append((f"Date: {date}", 'normal'))
    lines.append((f"Doctor: {doctor}", 'normal'))
    lines.append(("Rx", 'medium'))

    for mi in range(1, num_meds + 1):
        med = random.choice(MEDICINES)
        dosage = random.choice(med['dosages'])
        form = random.choice(med['forms'])
        freq = random.choice(FREQUENCIES)
        dur = random.choice(DURATIONS)
        lines.append((f"{mi}. {med['name']} {dosage} {form}", 'normal'))
        lines.append((f"   {freq}", 'small'))
        lines.append((f"   Duration: {dur}", 'small'))

    lines.append((f"Signature: {doctor}", 'normal'))

    quality = pick_quality()
    quality_counts[quality] += 1
    fonts = random.choice(font_sets)

    for text, font_key in lines:
        line_count += 1
        filename = f"line_{line_count:06d}.png"
        img = render_line(text, fonts[font_key])
        img = degrade_image(img, quality)
        img.save(train_dir / filename)
        all_samples.append((filename, text.strip()))

    if rx_idx % 100 == 0:
        print(f"   {rx_idx}/{NUM_PRESCRIPTIONS} ({line_count} lines)")

# Shuffle and split 90/10
random.seed(42)
random.shuffle(all_samples)
split_idx = int(0.9 * len(all_samples))
train_samples = all_samples[:split_idx]
val_samples = all_samples[split_idx:]

# Move val images
for filename, _ in val_samples:
    src = train_dir / filename
    dst = val_dir / filename
    if src.exists():
        src.rename(dst)

# Write label files
with open(TRAINING_DATA_DIR / 'train_label.txt', 'w', encoding='utf-8') as f:
    for filename, label in train_samples:
        f.write(f"train/{filename}\t{label}\n")

with open(TRAINING_DATA_DIR / 'val_label.txt', 'w', encoding='utf-8') as f:
    for filename, label in val_samples:
        f.write(f"val/{filename}\t{label}\n")

# Build character dictionary
all_chars = set()
for _, label in all_samples:
    all_chars.update(label)
with open(TRAINING_DATA_DIR / 'en_dict.txt', 'w', encoding='utf-8') as f:
    for ch in sorted(all_chars):
        f.write(f"{ch}\n")

print(f"\nDone! Total: {line_count} lines")
print(f"  Train: {len(train_samples)}, Val: {len(val_samples)}")
print(f"  Dictionary: {len(all_chars)} characters")
print(f"  Quality: {quality_counts}")

In [ ]:
# Preview some training samples
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 1, figsize=(14, 8))
for i, ax in enumerate(axes):
    fname, label = train_samples[i]
    img = Image.open(train_dir / fname)
    ax.imshow(img)
    ax.set_title(f'Label: {label}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Download Pretrained Model & Create Config

In [ ]:
# Download PP-OCRv4 English recognition pretrained model
PRETRAINED_DIR = Path(WORK_DIR) / 'pretrained'
PRETRAINED_DIR.mkdir(exist_ok=True)
MODEL_NAME = 'en_PP-OCRv4_rec_train'
PRETRAINED_URL = f'https://paddleocr.bj.bcebos.com/PP-OCRv4/english/{MODEL_NAME}.tar'

model_dir = PRETRAINED_DIR / MODEL_NAME
if not model_dir.exists():
    !wget -q -O {PRETRAINED_DIR}/{MODEL_NAME}.tar {PRETRAINED_URL}
    !tar -xf {PRETRAINED_DIR}/{MODEL_NAME}.tar -C {PRETRAINED_DIR}
    !rm {PRETRAINED_DIR}/{MODEL_NAME}.tar
    print(f"Downloaded pretrained model to {model_dir}")
else:
    print(f"Pretrained model exists at {model_dir}")

!ls {model_dir}

In [ ]:
import yaml

# Training hyperparameters
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.0005

OUTPUT_DIR = Path(WORK_DIR) / 'model'
OUTPUT_DIR.mkdir(exist_ok=True)

# Count dictionary characters
dict_file = TRAINING_DATA_DIR / 'en_dict.txt'
with open(dict_file, 'r') as f:
    num_chars = len(f.read().strip().split('\n'))
num_classes = num_chars + 2  # +2 for blank and unknown tokens

config = {
    'Global': {
        'debug': False,
        'use_gpu': True,
        'epoch_num': EPOCHS,
        'log_smooth_window': 20,
        'print_batch_step': 10,
        'save_model_dir': str(OUTPUT_DIR / 'training_output'),
        'save_epoch_step': 10,
        'eval_batch_step': [0, 200],
        'cal_metric_during_train': True,
        'pretrained_model': str(model_dir / 'best_accuracy'),
        'checkpoints': None,
        'save_inference_dir': str(OUTPUT_DIR / 'inference'),
        'use_visualdl': False,
        'infer_img': None,
        'character_dict_path': str(dict_file),
        'max_text_length': 80,
        'infer_mode': False,
        'use_space_char': True,
        'distributed': False,
        'save_res_path': str(OUTPUT_DIR / 'rec_results.txt'),
    },
    'Optimizer': {
        'name': 'Adam',
        'beta1': 0.9,
        'beta2': 0.999,
        'lr': {
            'name': 'Cosine',
            'learning_rate': LEARNING_RATE,
            'warmup_epoch': 5,
        },
        'regularizer': {
            'name': 'L2',
            'factor': 3.0e-5,
        },
    },
    'Architecture': {
        'model_type': 'rec',
        'algorithm': 'SVTR_LCNet',
        'Transform': None,
        'Backbone': {
            'name': 'PPLCNetV3',
            'scale': 0.95,
        },
        'Head': {
            'name': 'MultiHead',
            'head_list': [
                {
                    'CTCHead': {
                        'Neck': {
                            'name': 'svtr',
                            'dims': 120,
                            'depth': 2,
                            'hidden_dims': 120,
                            'kernel_size': [1, 3],
                            'use_guide': True,
                        },
                        'Head': {
                            'fc_decay': 1.0e-5,
                        },
                    }
                },
                {
                    'NRTRHead': {
                        'nrtr_dim': 384,
                        'max_text_length': 80,
                    }
                },
            ],
            'out_channels_list': {
                'CTCLabelDecode': num_classes,
                'NRTRLabelDecode': num_classes,
            },
        },
    },
    'Loss': {
        'name': 'MultiLoss',
        'loss_config_list': [
            {'CTCLoss': None},
            {'NRTRLoss': None},
        ],
    },
    'PostProcess': {
        'name': 'CTCLabelDecode',
    },
    'Metric': {
        'name': 'RecMetric',
        'main_indicator': 'acc',
        'ignore_space': False,
    },
    'Train': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(TRAINING_DATA_DIR),
            'ext_op_transform_idx': 1,
            'label_file_list': [str(TRAINING_DATA_DIR / 'train_label.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'RecConAug': {
                    'prob': 0.5,
                    'ext_data_num': 2,
                    'image_shape': [48, 320, 3],
                    'max_text_length': 80,
                }},
                {'RecAug': None},
                {'MultiLabelEncode': {
                    'gtc_encode': 'NRTRLabelEncode',
                }},
                {'RecResizeImg': {'image_shape': [3, 48, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
            ],
        },
        'loader': {
            'shuffle': True,
            'batch_size_per_card': BATCH_SIZE,
            'drop_last': True,
            'num_workers': 4,
        },
    },
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(TRAINING_DATA_DIR),
            'label_file_list': [str(TRAINING_DATA_DIR / 'val_label.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'MultiLabelEncode': {
                    'gtc_encode': 'NRTRLabelEncode',
                }},
                {'RecResizeImg': {'image_shape': [3, 48, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
            ],
        },
        'loader': {
            'shuffle': False,
            'drop_last': False,
            'batch_size_per_card': BATCH_SIZE,
            'num_workers': 4,
        },
    },
}

config_path = OUTPUT_DIR / 'rec_prescription_train.yml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Config saved to {config_path}")
print(f"  Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"  Dictionary: {num_chars} chars, Classes: {num_classes}")
print(f"  GPU: True")

## 5. Train the Model

This will fine-tune PP-OCRv4 recognition on our prescription data. On a T4 GPU, expect ~15-30 minutes for 50 epochs.

In [ ]:
# Start training!
!cd {WORK_DIR}/PaddleOCR && python tools/train.py -c {config_path}

## 6. Export Inference Model

In [ ]:
# Find best checkpoint
import glob

training_output = OUTPUT_DIR / 'training_output'

# Debug: show what was actually saved
print("Looking for checkpoints in:", training_output)
if training_output.exists():
    for f in sorted(training_output.iterdir()):
        print(f"  {f.name}")
else:
    print("  Directory does not exist!")

# Also check if PaddleOCR saved to its default 'output/' dir in CWD
alt_output = Path(WORK_DIR) / 'PaddleOCR' / 'output'
if alt_output.exists():
    print(f"\nAlso found output at: {alt_output}")
    for f in sorted(alt_output.iterdir()):
        print(f"  {f.name}")

# Search for checkpoints in expected and fallback locations
checkpoint = None
for search_dir in [training_output, alt_output]:
    if not search_dir.exists():
        continue
    best = search_dir / 'best_accuracy'
    latest = search_dir / 'latest'
    if Path(str(best) + '.pdparams').exists():
        checkpoint = str(best)
        print(f"\nUsing best_accuracy from {search_dir}")
        break
    elif Path(str(latest) + '.pdparams').exists():
        checkpoint = str(latest)
        print(f"\nUsing latest from {search_dir}")
        break

# Last resort: search recursively for any .pdparams
if checkpoint is None:
    found = glob.glob(str(Path(WORK_DIR) / '**' / 'best_accuracy.pdparams'), recursive=True)
    if not found:
        found = glob.glob(str(Path(WORK_DIR) / '**' / 'latest.pdparams'), recursive=True)
    if found:
        checkpoint = found[0].replace('.pdparams', '')
        print(f"\nFound checkpoint via search: {checkpoint}")

if checkpoint is None:
    raise FileNotFoundError(
        f"No trained checkpoint found!\n"
        f"Searched: {training_output}, {alt_output}\n"
        f"Check training logs above for errors."
    )

inference_dir = OUTPUT_DIR / 'inference'
print(f"Exporting from: {checkpoint}")
print(f"Export to: {inference_dir}")

In [ ]:
import subprocess

# Run export and capture output
export_cmd = (
    f"cd {WORK_DIR}/PaddleOCR && python tools/export_model.py "
    f"-c {config_path} "
    f'-o Global.pretrained_model="{checkpoint}" '
    f'   Global.save_inference_dir="{inference_dir}"'
)
print(f"Running: {export_cmd}\n")
result = subprocess.run(export_cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

# Check if export actually created the files
if not inference_dir.exists():
    # PaddleOCR may save relative to its CWD instead of the absolute path
    alt_inference_dirs = [
        Path(WORK_DIR) / 'PaddleOCR' / 'inference',
        Path(WORK_DIR) / 'PaddleOCR' / 'output' / 'inference',
        Path(WORK_DIR) / 'PaddleOCR' / str(inference_dir).lstrip('/'),
    ]
    # Also search recursively for inference.pdmodel
    import glob as _glob
    found_models = _glob.glob(str(Path(WORK_DIR) / '**' / 'inference.pdmodel'), recursive=True)
    if found_models:
        alt_inference_dirs.insert(0, Path(found_models[0]).parent)

    for alt_dir in alt_inference_dirs:
        if alt_dir.exists() and any(alt_dir.glob('*.pdmodel')):
            print(f"\nExport went to unexpected location: {alt_dir}")
            print(f"Moving to expected location: {inference_dir}")
            import shutil
            inference_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(str(alt_dir), str(inference_dir))
            break

if inference_dir.exists() and any(inference_dir.glob('*.pdmodel')):
    print(f"\nExport successful! Files in {inference_dir}")
else:
    print(f"\nERROR: Export failed — no .pdmodel found in {inference_dir}")
    print(f"Return code: {result.returncode}")
    print("Check the output above for errors.")

In [ ]:
# Verify exported files
if inference_dir.exists() and any(inference_dir.iterdir()):
    print("Exported model files:")
    for f in sorted(inference_dir.iterdir()):
        size_mb = f.stat().st_size / (1024*1024)
        print(f"  {f.name}: {size_mb:.1f} MB")
else:
    print(f"ERROR: {inference_dir} does not exist or is empty!")
    print(f"\nSearching for exported model files anywhere under {WORK_DIR}...")
    import glob as _glob
    for pattern in ['inference.pdmodel', 'inference.pdiparams']:
        hits = _glob.glob(str(Path(WORK_DIR) / '**' / pattern), recursive=True)
        for h in hits:
            print(f"  Found: {h} ({Path(h).stat().st_size / (1024*1024):.1f} MB)")
    if not hits:
        print("  No exported model files found. Re-run the export cell above.")

## 7. Quick Test

In [ ]:
# Test the trained model on a few validation samples
from paddleocr import PaddleOCR
import matplotlib.pyplot as plt

# Load with custom model
ocr = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    show_log=False,
    rec_model_dir=str(inference_dir),
    rec_char_dict_path=str(dict_file),
)

# Test on 5 random val images
val_files = list(val_dir.glob('*.png'))[:5]
fig, axes = plt.subplots(5, 1, figsize=(14, 8))

for i, img_path in enumerate(val_files):
    result = ocr.ocr(str(img_path), cls=True)
    if result and result[0]:
        text = result[0][0][1][0]
        conf = result[0][0][1][1]
    else:
        text, conf = '(empty)', 0.0

    # Find ground truth
    gt = next((l for f, l in val_samples if f == img_path.name), '?')

    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f'GT: {gt}\nOCR [{conf:.3f}]: {text}', fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 8. Download Trained Model

Download the model files to use in your local project. You need **3 files**:
- `inference.pdmodel` — model architecture
- `inference.pdiparams` — trained weights
- `inference.pdiparams.info` — parameter info

Also download `en_dict.txt` (character dictionary).

**Place them in your project:**
```
ml-models/prescription-ocr/model/inference/
    inference.pdmodel
    inference.pdiparams
    inference.pdiparams.info
ml-models/prescription-ocr/training_data/
    en_dict.txt
```

In [ ]:
# Zip everything needed for deployment
import shutil

export_dir = Path(WORK_DIR) / 'export_for_local'
export_dir.mkdir(exist_ok=True)

# Copy inference model
export_inference = export_dir / 'inference'
if export_inference.exists():
    shutil.rmtree(export_inference)
shutil.copytree(inference_dir, export_inference)

# Copy dictionary
shutil.copy(dict_file, export_dir / 'en_dict.txt')

# Zip it
zip_path = Path(WORK_DIR) / 'trained_prescription_model'
shutil.make_archive(str(zip_path), 'zip', str(export_dir))
print(f"Zip created: {zip_path}.zip")
print(f"Size: {Path(str(zip_path) + '.zip').stat().st_size / (1024*1024):.1f} MB")

In [ ]:
# Download the zip file
from google.colab import files
files.download(str(zip_path) + '.zip')

## 9. After Download — Local Setup

Once downloaded, extract the zip on your local machine:

```
1. Extract trained_prescription_model.zip
2. Copy inference/ folder → ml-models/prescription-ocr/model/inference/
3. Copy en_dict.txt → ml-models/prescription-ocr/training_data/en_dict.txt
4. Start the API: python api_service.py
   → It will auto-detect the custom model and load it!
```

The `api_service.py` is already configured to:
- Check `model/inference/` for `.pdmodel` files
- If found → loads your custom trained model
- If not found → uses pretrained PaddleOCR model

The API response will show `modelType: 'paddleocr-custom'` when using your trained model.